In [3]:
import pandas as pd


df = pd.read_csv('/Users/pranavipenumala/Downloads/data_project.csv')

print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Age                       1470 non-null   int64
 1   Attrition                 1470 non-null   str  
 2   BusinessTravel            1470 non-null   str  
 3   DailyRate                 1470 non-null   int64
 4   Department                1470 non-null   str  
 5   DistanceFromHome          1470 non-null   int64
 6   Education                 1470 non-null   int64
 7   EducationField            1470 non-null   str  
 8   EmployeeCount             1470 non-null   int64
 9   EmployeeNumber            1470 non-null   int64
 10  EnvironmentSatisfaction   1470 non-null   int64
 11  Gender                    1470 non-null   str  
 12  HourlyRate                1470 non-null   int64
 13  JobInvolvement            1470 non-null   int64
 14  JobLevel                  1470 non-null   int64
 15

In [4]:
import numpy as np
import pandas as pd

# ==========================================
# 1. LOAD THE DATASET
# ==========================================
file_path = '/Users/pranavipenumala/Downloads/data_project.csv'
df = pd.read_csv(file_path)

print('--- RAW DATA OVERVIEW ---')
print(f'Initial Shape: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Total Missing Values: {df.isnull().sum().sum()}')
print(f'Total Duplicate Rows: {df.duplicated().sum()}\n')

# ==========================================
# 2. CREATE DATA DICTIONARY (Guidelines 1 & 3)
# ==========================================
data_dict = pd.DataFrame({
    'Column_Name': df.columns,
    'Data_Type': df.dtypes.astype(str).values,
    'Unique_Count': [df[col].nunique() for col in df.columns],
    'Variable_Type': np.where(
        df.dtypes.isin(['int64', 'float64']), 'Numerical', 'Categorical'
    ),
})
data_dict.to_csv('Data_Dictionary.csv', index=False)
print('✓ Saved: Data_Dictionary.csv')

# ==========================================
# 3. DATA CLEANING & PREPARATION (Guideline 2)
# ==========================================
# Remove constant and identifier columns that provide no analytical value
cols_to_drop = ['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber']
df_clean = df.drop(columns=[col for col in cols_to_drop if col in df.columns]).copy()

# Remove duplicate rows if any exist
df_clean = df_clean.drop_duplicates()

# Encode target Attrition variable into binary (1 = Yes, 0 = No)
df_clean['Attrition_Flag'] = np.where(
    df_clean['Attrition'].str.strip().str.lower() == 'yes', 1, 0
)

# Feature Engineering: Binning continuous variables for dashboard slicing
df_clean['AgeGroup'] = pd.cut(
    df_clean['Age'],
    bins=[17, 25, 35, 45, 65],
    labels=['18-25', '26-35', '36-45', '46+'],
)

df_clean['SalarySlab'] = pd.cut(
    df_clean['MonthlyIncome'],
    bins=[0, 3000, 6000, 10000, 25000],
    labels=['<3k', '3k-6k', '6k-10k', '10k+'],
)

df_clean['TenureBand'] = pd.cut(
    df_clean['YearsAtCompany'],
    bins=[-1, 2, 5, 10, 45],
    labels=['0-2 yrs', '3-5 yrs', '6-10 yrs', '10+ yrs'],
)

# Save Cleaned Dataset for Dashboard (Power BI / Tableau)
df_clean.to_csv('cleaned_employee_attrition.csv', index=False)
print(
    f'✓ Saved: cleaned_employee_attrition.csv with shape {df_clean.shape}\n'
)

# ==========================================
# 4. KEY HR METRICS / KPIS (Guideline 5)
# ==========================================
total_headcount = len(df_clean)
total_exits = int(df_clean['Attrition_Flag'].sum())
overall_attrition_rate = (total_exits / total_headcount) * 100

avg_income_all = df_clean['MonthlyIncome'].mean()
avg_income_leavers = df_clean.loc[
    df_clean['Attrition_Flag'] == 1, 'MonthlyIncome'
].mean()
avg_income_stayers = df_clean.loc[
    df_clean['Attrition_Flag'] == 0, 'MonthlyIncome'
].mean()

avg_tenure_all = df_clean['YearsAtCompany'].mean()
avg_tenure_leavers = df_clean.loc[
    df_clean['Attrition_Flag'] == 1, 'YearsAtCompany'
].mean()
avg_tenure_stayers = df_clean.loc[
    df_clean['Attrition_Flag'] == 0, 'YearsAtCompany'
].mean()

kpis = pd.DataFrame({
    'Metric': [
        'Total Headcount',
        'Total Exits',
        'Overall Attrition Rate (%)',
        'Avg Monthly Income (Overall)',
        'Avg Monthly Income (Stayers)',
        'Avg Monthly Income (Leavers)',
        'Avg Tenure Years (Overall)',
        'Avg Tenure Years (Stayers)',
        'Avg Tenure Years (Leavers)',
    ],
    'Value': [
        f'{total_headcount}',
        f'{total_exits}',
        f'{overall_attrition_rate:.2f}%',
        f'${avg_income_all:,.2f}',
        f'${avg_income_stayers:,.2f}',
        f'${avg_income_leavers:,.2f}',
        f'{avg_tenure_all:.2f} yrs',
        f'{avg_tenure_stayers:.2f} yrs',
        f'{avg_tenure_leavers:.2f} yrs',
    ],
})
print('--- KEY HR METRICS ---')
print(kpis.to_string(index=False))
print('\n')


# ==========================================
# 5. SEGMENTED ATTRITION & DRIVERS (Guidelines 6 & 7)
# ==========================================
def calculate_turnover(dataframe, column_name):
  summary = dataframe.groupby(column_name, observed=False).agg(
      Total_Employees=('Attrition_Flag', 'count'),
      Exits=('Attrition_Flag', 'sum'),
      Attrition_Rate=('Attrition_Flag', lambda x: np.round(x.mean() * 100, 2)),
  )
  return summary.sort_values(by='Attrition_Rate', ascending=False)


print('--- ATTRITION BY OVERTIME ---')
print(calculate_turnover(df_clean, 'OverTime'))
print('\n--- ATTRITION BY JOB ROLE ---')
print(calculate_turnover(df_clean, 'JobRole'))
print('\n--- ATTRITION BY DEPARTMENT ---')
print(calculate_turnover(df_clean, 'Department'))
print('\n--- ATTRITION BY SALARY SLAB ---')
print(calculate_turnover(df_clean, 'SalarySlab'))
print('\n--- ATTRITION BY WORK-LIFE BALANCE (1=Bad, 4=Best) ---')
print(calculate_turnover(df_clean, 'WorkLifeBalance'))

--- RAW DATA OVERVIEW ---
Initial Shape: 1470 rows, 35 columns
Total Missing Values: 0
Total Duplicate Rows: 0

✓ Saved: Data_Dictionary.csv
✓ Saved: cleaned_employee_attrition.csv with shape (1470, 35)

--- KEY HR METRICS ---
                      Metric     Value
             Total Headcount      1470
                 Total Exits       237
  Overall Attrition Rate (%)    16.12%
Avg Monthly Income (Overall) $6,502.93
Avg Monthly Income (Stayers) $6,832.74
Avg Monthly Income (Leavers) $4,787.09
  Avg Tenure Years (Overall)  7.01 yrs
  Avg Tenure Years (Stayers)  7.37 yrs
  Avg Tenure Years (Leavers)  5.13 yrs


--- ATTRITION BY OVERTIME ---
          Total_Employees  Exits  Attrition_Rate
OverTime                                        
Yes                   416    127           30.53
No                   1054    110           10.44

--- ATTRITION BY JOB ROLE ---
                           Total_Employees  Exits  Attrition_Rate
JobRole                                                   

In [5]:
import numpy as np
import pandas as pd

# Load the cleaned dataset
df = pd.read_csv('cleaned_employee_attrition.csv')

print('=' * 60)
print('EXPLORATORY DATA ANALYSIS (EDA) REPORT METRICS')
print('=' * 60)

# ------------------------------------------------------------
# 1. OVERALL METRICS / EXECUTIVE SUMMARY (Guideline 5)
# ------------------------------------------------------------
total_emp = len(df)
exited_emp = int(df['Attrition_Flag'].sum())
stayed_emp = total_emp - exited_emp
turnover_rate = (exited_emp / total_emp) * 100

print(f'Total Headcount       : {total_emp}')
print(f'Stayed Employees      : {stayed_emp} ({stayed_emp/total_emp*100:.2f}%)')
print(f'Exited Employees      : {exited_emp} ({turnover_rate:.2f}%)')
print(f'Overall Turnover Rate : {turnover_rate:.2f}%\n')

# ------------------------------------------------------------
# 2. STAYERS VS LEAVERS PROFILE COMPARISON
# ------------------------------------------------------------
compare_cols = [
    'Age',
    'MonthlyIncome',
    'TotalWorkingYears',
    'YearsAtCompany',
    'YearsInCurrentRole',
    'YearsWithCurrManager',
]
comparison_table = df.groupby('Attrition')[compare_cols].mean().round(2).T
comparison_table.columns = ['Stayers (No)', 'Leavers (Yes)']

print('=' * 60)
print('AVERAGE METRIC VALUES: STAYERS VS LEAVERS')
print('=' * 60)
print(comparison_table)
print('\n')


# ------------------------------------------------------------
# HELPER FUNCTION FOR SEGMENTED ANALYSIS (Guidelines 6 & 7)
# ------------------------------------------------------------
def generate_segment_table(dataframe, group_col):
  table = dataframe.groupby(group_col, observed=False)['Attrition_Flag'].agg(
      Headcount='count',
      Exits='sum',
      Attrition_Rate=lambda x: np.round(x.mean() * 100, 2),
  )
  return table.sort_values(by='Attrition_Rate', ascending=False)


# ------------------------------------------------------------
# 3. OVERTIME IMPACT (Guideline 7)
# ------------------------------------------------------------
print('=' * 60)
print('ATTRITION BY OVERTIME')
print('=' * 60)
print(generate_segment_table(df, 'OverTime'))
print('\n')

# ------------------------------------------------------------
# 4. JOB ROLE BREAKDOWN (Guideline 6)
# ------------------------------------------------------------
print('=' * 60)
print('ATTRITION BY JOB ROLE')
print('=' * 60)
print(generate_segment_table(df, 'JobRole'))
print('\n')

# ------------------------------------------------------------
# 5. DEPARTMENT BREAKDOWN (Guideline 6)
# ------------------------------------------------------------
print('=' * 60)
print('ATTRITION BY DEPARTMENT')
print('=' * 60)
print(generate_segment_table(df, 'Department'))
print('\n')

# ------------------------------------------------------------
# 6. SALARY SLAB BREAKDOWN (Guideline 7)
# ------------------------------------------------------------
print('=' * 60)
print('ATTRITION BY SALARY SLAB')
print('=' * 60)
print(generate_segment_table(df, 'SalarySlab'))
print('\n')

# ------------------------------------------------------------
# 7. WORK-LIFE BALANCE & SATISFACTION BREAKDOWN (Guideline 7)
# ------------------------------------------------------------
print('=' * 60)
print('ATTRITION BY WORK-LIFE BALANCE (1=Bad, 4=Best)')
print('=' * 60)
print(generate_segment_table(df, 'WorkLifeBalance'))
print('\n')

print('=' * 60)
print('ATTRITION BY JOB SATISFACTION (1=Low, 4=High)')
print('=' * 60)
print(generate_segment_table(df, 'JobSatisfaction'))
print('\n')

# ------------------------------------------------------------
# 8. AGE GROUP BREAKDOWN (Guideline 6)
# ------------------------------------------------------------
print('=' * 60)
print('ATTRITION BY AGE GROUP')
print('=' * 60)
print(generate_segment_table(df, 'AgeGroup'))

EXPLORATORY DATA ANALYSIS (EDA) REPORT METRICS
Total Headcount       : 1470
Stayed Employees      : 1233 (83.88%)
Exited Employees      : 237 (16.12%)
Overall Turnover Rate : 16.12%

AVERAGE METRIC VALUES: STAYERS VS LEAVERS
                      Stayers (No)  Leavers (Yes)
Age                          37.56          33.61
MonthlyIncome              6832.74        4787.09
TotalWorkingYears            11.86           8.24
YearsAtCompany                7.37           5.13
YearsInCurrentRole            4.48           2.90
YearsWithCurrManager          4.37           2.85


ATTRITION BY OVERTIME
          Headcount  Exits  Attrition_Rate
OverTime                                  
Yes             416    127           30.53
No             1054    110           10.44


ATTRITION BY JOB ROLE
                           Headcount  Exits  Attrition_Rate
JobRole                                                    
Sales Representative              83     33           39.76
Laboratory Technician    

In [7]:
report_content = """# Employee Attrition Analysis Project

## Deliverable 1: Data Cleaning & Preparation Report

### 1. Dataset Overview & Audit
- Initial Shape: 1,470 rows, 35 columns
- Total Missing Values: 0
- Total Duplicate Rows: 0

### 2. Redundant Columns Removed
The following columns were dropped because they have zero variance or are arbitrary identifiers:
- EmployeeCount (equal to 1 for all rows)
- Over18 (equal to 'Y' for all rows)
- StandardHours (equal to 80 for all rows)
- EmployeeNumber (arbitrary identifier)

### 3. Feature Engineering
- Attrition_Flag: Encoded Attrition into numeric (1 for Yes, 0 for No).
- AgeGroup: Binned into '18-25', '26-35', '36-45', '46+'.
- SalarySlab: Binned MonthlyIncome into '<3k', '3k-6k', '6k-10k', '10k+'.
- TenureBand: Binned YearsAtCompany into '0-2 yrs', '3-5 yrs', '6-10 yrs', '10+ yrs'.

### 4. Output Files Created
- cleaned_employee_attrition.csv: 1,470 rows, 35 columns (ready for dashboard).
- Data_Dictionary.csv: Metadata for all columns.

---

## Deliverable 2: Exploratory Data Analysis (EDA) Report

### 1. Key Performance Indicators (KPIs)
- Total Headcount: 1,470
- Stayed Employees: 1,233 (83.88%)
- Exited Employees: 237 (16.12%)
- Overall Turnover Rate: 16.12%

### 2. Leavers vs. Stayers Profile
- Monthly Income: Leavers earn $4,787.09 vs. $6,832.74 for Stayers (~30% less).
- Company Tenure: Leavers average 5.13 years vs. 7.37 years for Stayers.
- Average Age: Leavers average 33.61 years vs. 37.56 years for Stayers.
- Years in Current Role: Leavers average 2.90 years vs. 4.48 years for Stayers.

### 3. Key Turnover Drivers
1. Overtime:
   - OverTime = Yes: 30.53% Attrition Rate (127 exits / 416 employees)
   - OverTime = No: 10.44% Attrition Rate (110 exits / 1,054 employees)
   - Conclusion: Overtime employees are ~3x more likely to leave.

2. Monthly Compensation:
   - Low (<$3k): 28.61% Attrition Rate (113 exits)
   - Medium ($3k-$6k): 12.72% Attrition Rate (66 exits)
   - High ($6k-$10k): 12.00% Attrition Rate (33 exits)
   - Executive (>$10k): 8.90% Attrition Rate (25 exits)
   - Conclusion: Turnover drops sharply once salary exceeds $3,000.

3. Work-Life Balance:
   - Level 1 (Poor): 31.25% Attrition Rate
   - Level 2: 16.86% Attrition Rate
   - Level 3: 14.22% Attrition Rate
   - Level 4: 17.65% Attrition Rate

### 4. Segment Breakdown
- High-Risk Job Roles:
  * Sales Representative: 39.76% (33 exits / 83 employees)
  * Laboratory Technician: 23.94% (62 exits / 259 employees)
  * Human Resources: 23.08% (12 exits / 52 employees)
- Low-Risk Job Roles:
  * Research Director: 2.50%
  * Manager: 4.90%
- Department Turnover:
  * Sales: 20.63%
  * Human Resources: 19.05%
  * Research & Development: 13.84%
- Age Demographics:
  * 18-25: 35.77% Attrition Rate (highest risk)
  * 26-35: 19.14%
  * 46+: 12.45%
  * 36-45: 9.19% (most stable)

---

## Deliverable 4: Business Insights & Recommendations

1. Overtime Cap & Burnout Management:
   - Restrict mandatory overtime hours, specifically in high-risk departments (Sales and Laboratory).
   - Provide compensatory off-days or overtime premiums.

2. Compensation Restructuring for Entry-Level Roles:
   - Review market benchmark salaries for Sales Representatives and Lab Technicians earning under $3,000/month.

3. Early-Career Mentorship:
   - Implement structured 90-day and 1-year career pathing reviews for hires aged 18-25 and employees with under 2 years of tenure.
"""

with open("Employee_Attrition_Report.txt", "w") as f:
  f.write(report_content)

print("Report saved successfully as 'Employee_Attrition_Report.txt'!")

Report saved successfully as 'Employee_Attrition_Report.txt'!
